# NB03 — Entraînement LSTM / Transformer sur Colab GPU (Semaine 3)

**Objectif** : Lancer les 6 runs (LSTM × 3 seeds + Transformer × 3 seeds) sur les splits verrouillés v02, avec persistance sur Google Drive.

**Prérequis** :
- Compte Google + Google Drive activé
- Repo `Projet_Peptide` sur GitHub (public ou token PAT si privé)
- Branche `main` à jour : contient `data/splits/*_v02.csv` + `configs/*_base.yaml` (schéma imbriqué)

**Durée estimée Colab GPU T4** :
- LSTM : ~15-20 min / run × 3 = 45-60 min
- Transformer : ~30-45 min / run × 3 = 90-135 min
- **Total ≈ 2-3 h** → une seule session

In [6]:
# 1. Monter Google Drive (persistance)
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# Dossier de travail persistant sur Drive
import os, sys, subprocess, json, time, shutil
WORK = '/content/drive/MyDrive/Projet_Peptide'
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, WORK)
print(f'Travail dans : {WORK}')
print(f'Contenu : {os.listdir(WORK)}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Travail dans : /content/drive/MyDrive/Projet_Peptide
Contenu : ['Projet_Peptide', 'checkpoints', 'results']


In [7]:
# 2. Cloner / mettre à jour le repo
REPO_URL = 'https://github.com/FEDI-HASSINE/Projet_Peptide.git'
REPO_DIR = os.path.join(WORK, 'Projet_Peptide')

if not os.path.exists(os.path.join(REPO_DIR, '.git')):
    print('Clone du repo...')
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print('Repo existant → fetch + checkout main')
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', 'main'], check=True)

os.chdir(REPO_DIR)
print(f'Dans {REPO_DIR}')
print(f'Branche : {subprocess.run(["git", "branch", "--show-current"], capture_output=True, text=True, cwd=REPO_DIR).stdout.strip()}')
print(f'Commit : {subprocess.run(["git", "rev-parse", "--short=7", "HEAD"], capture_output=True, text=True, cwd=REPO_DIR).stdout.strip()}')

Repo existant → fetch + checkout main
Dans /content/drive/MyDrive/Projet_Peptide/Projet_Peptide
Branche : main
Commit : df7f493


In [8]:
# 3. Vérifier les artefacts verrouillés (splits v02 + configs)
required = [
    'data/splits/train_v02.csv',
    'data/splits/val_v02.csv',
    'data/splits/test_v02.csv',
    'configs/lstm_base.yaml',
    'configs/transformer_base.yaml',
    'requirements.txt',
]
missing = [f for f in required if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(f'Artefacts manquants sur main : {missing}')
print('Tous les artefacts verrouillés présents')

# Afficher tailles splits
import pandas as pd
for split in ['train', 'val', 'test']:
    df = pd.read_csv(f'data/splits/{split}_v02.csv')
    print(f'   {split}_v02.csv : {len(df):,} séquences')

Tous les artefacts verrouillés présents
   train_v02.csv : 38,336 séquences
   val_v02.csv : 2,095 séquences
   test_v02.csv : 2,144 séquences


In [9]:
# 4. Installer les dépendances + vérifier GPU
print('Installation requirements...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'], check=True)

# Ajouter le VRAI dossier du repo (celui qui contient peplib/) au sys.path
REPO_DIR = '/content/drive/MyDrive/Projet_Peptide/Projet_Peptide'
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f'peplib accessible via sys.path : {REPO_DIR}')

# IMPORTANT : CUBLAS_WORKSPACE_CONFIG doit être posé AVANT import torch
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

# Vérifier torch + CUDA
import torch
print(f'Python : {sys.version.split()[0]}')
print(f'Torch  : {torch.__version__}')
print(f'CUDA   : {torch.cuda.is_available()}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if device == 'cpu':
    print('ATTENTION : pas de GPU détecté ! Runtime -> Changer le type d\'exécution -> GPU T4')

# Imports peplib (après torch pour seed deterministe)
from peplib.seed import set_seed
from peplib.traceability import run_metadata, git_commit, git_is_dirty, config_sha256, environment_info, utc_timestamp
from peplib.train import train_model
from peplib.generate import generate_with_stats
from peplib.metrics_proximity import proximity_details
from peplib.descriptors import (
    calculate_charge, calculate_gravy, calculate_hydrophobic_moment,
    calculate_pi, calculate_boman_index
)

Installation requirements...
peplib accessible via sys.path : /content/drive/MyDrive/Projet_Peptide/Projet_Peptide
Python : 3.13.16
Torch  : 2.14.1+cu130
CUDA   : True
Device : cuda


ModuleNotFoundError: No module named 'peplib.traceability'

In [ ]:
# 5. Configuration des runs
RUNS = [
    {'model': 'lstm', 'seed': 42, 'config': 'configs/lstm_base.yaml'},
    {'model': 'lstm', 'seed': 43, 'config': 'configs/lstm_base.yaml'},
    {'model': 'lstm', 'seed': 44, 'config': 'configs/lstm_base.yaml'},
    {'model': 'transformer', 'seed': 42, 'config': 'configs/transformer_base.yaml'},
    {'model': 'transformer', 'seed': 43, 'config': 'configs/transformer_base.yaml'},
    {'model': 'transformer', 'seed': 44, 'config': 'configs/transformer_base.yaml'},
]

# Dossier du repo (où train.py écrit runs.csv et checkpoints)
CKPT_BASE = os.path.join(REPO_DIR, 'checkpoints')
RESULTS_DIR = os.path.join(REPO_DIR, 'results', 'week3')
os.makedirs(CKPT_BASE, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
RUNS_CSV = os.path.join(RESULTS_DIR, 'runs.csv')

print(f'Checkpoints -> {CKPT_BASE}')
print(f'Runs CSV    -> {RUNS_CSV}')
print(f'{len(RUNS)} runs planifiés')

In [ ]:
# 6. Fonction d'entraînement unique avec resume + logging + traçabilité
def run_one(model_name, seed, config_path, resume=True):
    """Lance un entraînement inline, gère resume, écrit runs.csv avec traçabilité complète."""
    import yaml
    with open(config_path, encoding='utf-8') as f:
        cfg = yaml.safe_load(f)
    
    # Métadonnées de traçabilité (écrites sur chaque ligne du CSV)
    meta = run_metadata(cfg, seed=seed, device=device)
    
    print(f'{model_name.upper()} seed={seed} | device={device}')
    
    try:
        result = train_model(
            config_path=config_path,
            seed=seed,
            resume=resume,
        )
        # Afficher résumé
        print(f'   DONE : best epoch={result["best_epoch"]} | val_pp={result["best_val_pp"]:.2f}')
        return True
    except Exception as e:
        print(f'   ERREUR : {e}')
        return False

In [ ]:
# 7. Boucle principale — lancer les runs séquentiellement
from tqdm.auto import tqdm

print('='*60)
print('DEBUT DES 6 RUNS')
print('='*60)

success = []
failed = []

for run in tqdm(RUNS, desc='Runs globaux'):
    ok = run_one(run['model'], run['seed'], run['config'])
    if ok:
        success.append(f"{run['model']}-{run['seed']}")
    else:
        failed.append(f"{run['model']}-{run['seed']}")
    # Pause pour laisser respirer le GPU
    time.sleep(5)

print('\n' + '='*60)
print('RESUME')
print('='*60)
print(f'Reussis ({len(success)}) : {success}')
print(f'Echoues ({len(failed)}) : {failed}')

In [ ]:
# 8. Génération + évaluation sur checkpoints entraînés (P0-2)
import random, math
from pathlib import Path
import yaml

def load_train_sequences(csv_path):
    df = pd.read_csv(csv_path, dtype=str)
    raw = df['sequence'].dropna().astype(str).str.strip().str.upper().tolist()
    from peplib.filters import classify_sequence
    return [s for s in raw if classify_sequence(s) is None]

def copy_analysis(generated, train):
    train_set = set(train)
    exact = [s for s in generated if s in train_set]
    prefix = []
    for s in generated:
        if len(s) >= 10 and s not in train_set:
            for t in train:
                if t.startswith(s) and t != s:
                    prefix.append(s)
                    break
    return {
        'n_exact': len(exact), 'taux_exact': len(exact)/len(generated) if generated else 0,
        'n_prefix': len(prefix), 'taux_prefix': len(prefix)/len(generated) if generated else 0,
        'exemples_exact': exact[:3], 'exemples_prefix': prefix[:3]
    }

def wilson(success, n, z=1.96):
    if n == 0: return (0.0, 0.0)
    p = success / n
    denom = 1 + z*z/n
    center = (p + z*z/(2*n)) / denom
    margin = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / denom
    return (max(0.0, center-margin), min(1.0, center+margin))

train_seqs = load_train_sequences('data/splits/train_v02.csv')
print(f'Train sequences valides : {len(train_seqs)}')

from peplib.train import build_model_from_config
from peplib.generate import generate_with_stats
from peplib.metrics_proximity import proximity_details
from peplib.descriptors import (
    calculate_charge, calculate_gravy, calculate_hydrophobic_moment,
    calculate_pi, calculate_boman_index
)

gen_results = []

for run in RUNS:
    model_name = run['model']
    seed = run['seed']
    config_path = run['config']
    
    # Charger config + checkpoint
    with open(config_path, encoding='utf-8') as f:
        cfg = yaml.safe_load(f)
    
    ckpt_path = Path('checkpoints') / model_name / f'seed_{seed}' / 'best.pt'
    if not ckpt_path.exists():
        print(f'{model_name} seed {seed} : checkpoint introuvable -> skip')
        continue
    
    print(f'\n--- Génération {model_name} seed {seed} ---')
    
    # Modèle + checkpoint
    model = build_model_from_config(cfg).to(device)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    model.eval()
    
    # Génération avec stats brutes (P0-2)
    stats = generate_with_stats(
        model, n=1000, temperature=1.0, top_k=0, top_p=1.0,
        max_len=52, seed=seed, device=device, max_attempts=15000
    )
    sequences = stats['sequences']
    
    print(f'  Tentatives={stats["n_attempts"]} | Valides={stats["n_valid"]} | ' 
          f'Taux_validité_brut={stats["n_valid"]/stats["n_attempts"]:.3f}')
    print(f'  Rejets={stats["rejections"]}')
    
    if not sequences:
        print('  AUCUNE séquence valide générée')
        continue
    
    # Copies exactes / prefixe
    copies = copy_analysis(sequences, train_seqs)
    print(f'  Copies exactes={copies["n_exact"]} ({copies["taux_exact"]:.2%}) | '
          f'Prefixes={copies["n_prefix"]} ({copies["taux_prefix"]:.2%})')
    if copies['exemples_exact']:
        print(f'    Exemples exactes: {copies["exemples_exact"]}')
    if copies['exemples_prefix']:
        print(f'    Exemples prefixe: {copies["exemples_prefix"]}')
    
    # Proximité Smith-Waterman sur sous-échantillon (P0-2)
    n_prox = min(200, len(sequences))
    if n_prox > 0:
        sample = random.Random(seed).sample(sequences, n_prox)
        details = proximity_details(sample, train_seqs, threshold=0.5)
        for thr in (0.5, 0.7, 0.9):
            count = int((details['best_min'] >= thr).sum())
            low, high = wilson(count, n_prox)
            print(f'  Proche>={thr} : {count}/{n_prox} = {count/n_prox:.3f} ' 
                  f'(IC95=[{low:.3f}, {high:.3f}])')
    
    # Propriétés physico-chimiques (proxys)
    props = {'charge': [], 'gravy': [], 'hmoment': [], 'pi': [], 'boman': []}
    for s in sequences:
        props['charge'].append(calculate_charge(s))
        props['gravy'].append(calculate_gravy(s))
        props['hmoment'].append(calculate_hydrophobic_moment(s))
        props['pi'].append(calculate_pi(s))
        props['boman'].append(calculate_boman_index(s))
    
    prop_summary = {}
    for k, v in props.items():
        if v:
            prop_summary[k] = sum(v)/len(v)
            print(f'  {k}: {prop_summary[k]:.3f}')
    
    # Stocker pour synthèse
    gen_results.append({
        'model': model_name, 'seed': seed,
        'n_attempts': stats['n_attempts'], 'n_valid': stats['n_valid'],
        'taux_validite_brut': stats['n_valid']/stats['n_attempts'],
        'taux_copie_exacte': copies['taux_exact'],
        'taux_copie_prefixe': copies['taux_prefix'],
        'proximite': {f'{thr}': int((details['best_min'] >= thr).sum())/n_prox 
                      for thr in (0.5, 0.7, 0.9)} if n_prox else {},
        'proprietes': prop_summary,
        'checkpoint_sha256': __import__('hashlib').sha256(ckpt_path.read_bytes()).hexdigest()[:16],
    })
    
    # Pause GPU
    time.sleep(3)

print('\n=== SYNTHÈSE GÉNÉRATION ===')
for r in gen_results:
    print(f"{r['model']} seed {r['seed']}: valides={r['n_valid']} taux_brut={r['taux_validite_brut']:.3f} "
          f"copie_exacte={r['taux_copie_exacte']:.2%} F_0.7={r['proximite'].get('0.7',0):.3f}")

In [ ]:
# 9. Régénérer la figure de perte depuis le VRAI runs.csv (P0-1)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

RUNS_CSV = os.path.join(RESULTS_DIR, 'runs.csv')
OUT_PNG = os.path.join(RESULTS_DIR, 'visuals', '2_courbes_perte.png')
os.makedirs(os.path.dirname(OUT_PNG), exist_ok=True)

if os.path.exists(RUNS_CSV):
    df = pd.read_csv(RUNS_CSV)
    
    # Vérifier colonnes requises
    required = ['model', 'seed', 'epoch', 'train_loss', 'val_loss', 'val_pp']
    missing = [c for c in required if c not in df.columns]
    if missing:
        print(f'Colonnes manquantes dans runs.csv : {missing}')
    else:
        models = sorted(df['model'].unique())
        colors = {'lstm': '#4c72b0', 'transformer': '#dd8452'}
        
        fig, axes = plt.subplots(1, len(models), figsize=(6*len(models), 4.4))
        if len(models) == 1:
            axes = [axes]
        
        for ax, model in zip(axes, models):
            subset = df[df['model'] == model]
            color = colors.get(model, '#555552')
            
            # Courbes par seed (transparentes)
            for seed in sorted(subset['seed'].unique()):
                per_seed = subset[subset['seed'] == seed].sort_values('epoch')
                ax.plot(per_seed['epoch'], per_seed['val_loss'],
                       color=color, alpha=0.45, linewidth=1.2)
            
            # Moyenne des seeds (forte)
            mean_curve = subset.groupby('epoch')['val_loss'].mean().reset_index()
            ax.plot(mean_curve['epoch'], mean_curve['val_loss'],
                   color=color, linewidth=2.4, marker='o', markersize=3,
                   label=f'{model.upper()} val loss (moyenne)')
            
            # Meilleure epoch (early stop)
            best = subset.loc[subset.groupby('seed')['val_loss'].idxmin()].sort_values('seed')
            first_best = best.iloc[0]
            ax.axvline(first_best['epoch'], color='black', linestyle='--', linewidth=1.0,
                      label=f'Meilleure epoch (seed {int(first_best["seed"])})')
            ax.plot(first_best['epoch'], first_best['val_loss'],
                   marker='*', color='black', markersize=12, zorder=5)
            
            ax.set_title(f'{model.upper()} -- Perte validation par seed', fontsize=12, fontweight='bold')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Cross-entropy (nats/token)')
            ax.grid(True, alpha=0.3)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.legend(fontsize=8)
        
        fig.suptitle('Courbes de perte reconstruites depuis runs.csv', fontsize=12)
        fig.tight_layout(rect=(0, 0, 1, 0.96))
        fig.savefig(OUT_PNG, dpi=200)
        plt.close(fig)
        
        print(f'✅ Figure régénérée : {OUT_PNG}')
        
        # Afficher les vraies valeurs
        for model in models:
            best = df[(df['model']==model)].loc[
                df[(df['model']==model)].groupby('seed')['val_loss'].idxmin()
            ]
            for _, row in best.iterrows():
                print(f"  {model} seed {int(row['seed'])}: epoch={int(row['epoch'])}"
                      f" val_loss={row['val_loss']:.4f} val_pp={row['val_pp']:.2f}")
else:
    print(f'❌ runs.csv introuvable : {RUNS_CSV}')

In [ ]:
# 10. Écrire section PROVENANCE.md Semaine 3 (P0-1)
from peplib.traceability import (
    upsert_markdown_block, render_week3_provenance,
    checkpoint_manifest, git_commit, git_is_dirty, utc_timestamp
)
import pandas as pd

RUNS_CSV = os.path.join(RESULTS_DIR, 'runs.csv')
GEN_SUMMARY = os.path.join(RESULTS_DIR, 'generation_summary.csv')
MANIFEST_CSV = os.path.join(RESULTS_DIR, 'checkpoints_sha256.csv')

# Manifeste des checkpoints
manifest = checkpoint_manifest('checkpoints')
manifest.to_csv(MANIFEST_CSV, index=False)
print(f'Checkpoint manifest : {MANIFEST_CSV} ({len(manifest)} fichiers)')

# Baselines de référence
from peplib.metrics_lm import positional_unigram_perplexity
train_for_baseline = pd.read_csv('data/splits/train_v02.csv', dtype=str)['sequence'].dropna().str.strip().str.upper().tolist()
val_for_baseline = pd.read_csv('data/splits/val_v02.csv', dtype=str)['sequence'].dropna().str.strip().str.upper().tolist()
pos_unigram_ppl = positional_unigram_perplexity(train_for_baseline, val_for_baseline, alpha=1.0)
unigram_ppl = 19.4874  # depuis baseline_a7.py

baselines = {
    'unigramme': unigram_ppl,
    'unigramme_positionnel': round(pos_unigram_ppl, 4),
}
print(f'Baselines : {baselines}')

# Génération summary
gen_df = pd.DataFrame()
if os.path.exists(GEN_SUMMARY):
    gen_df = pd.read_csv(GEN_SUMMARY)

# Runs
runs_df = pd.DataFrame()
if os.path.exists(RUNS_CSV):
    runs_df = pd.read_csv(RUNS_CSV)

# Rendu bloc markdown
block = render_week3_provenance(
    runs=runs_df,
    generation=gen_df,
    baselines=baselines,
    manifest_csv=MANIFEST_CSV,
    generated_at=utc_timestamp(),
)

upsert_markdown_block('PROVENANCE.md', block)
print('✅ PROVENANCE.md mis à jour (section Semaine 3)')

# Vérifier gate
if not runs_df.empty:
    gate = baselines['unigramme_positionnel']
    best = runs_df.loc[runs_df.groupby(['model', 'seed'])['val_pp'].idxmin()]
    summary = best.groupby('model')['val_pp'].mean()
    for model, mean_pp in summary.items():
        verdict = 'OK' if mean_pp < gate else 'ÉCHEC'
        print(f'  Gate {model}: val_pp_mean={mean_pp:.4f} < {gate:.4f} -> {verdict}')

In [ ]:
# 11. Résumé final + vérification
print('='*60)
print('RÉSUMÉ FINAL SEMAINE 3')
print('='*60)

# runs.csv
if os.path.exists(RUNS_CSV):
    df = pd.read_csv(RUNS_CSV)
    print(f'\nruns.csv : {len(df)} lignes')
    best = df.loc[df.groupby(['model', 'seed'])['val_pp'].idxmin()]
    print(best[['model', 'seed', 'epoch', 'val_pp', 'val_loss', 'elapsed_s']].to_string(index=False))
    agg = best.groupby('model')['val_pp'].agg(['mean', 'std', 'count']).round(4)
    print(f'\nAgrégation :')
    print(agg.to_string())

# generation_summary.csv
if os.path.exists(GEN_SUMMARY):
    gen = pd.read_csv(GEN_SUMMARY)
    print(f'\ngeneration_summary.csv : {len(gen)} lignes')
    print(gen[['model', 'seed', 'n_valid', 'taux_validite_brut', 'taux_copie_exacte', 'taux_copie_prefixe']].to_string(index=False))

# Figure
if os.path.exists(OUT_PNG):
    print(f'\n✅ Figure : {OUT_PNG}')
    # Afficher dans le notebook
    from IPython.display import Image, display
    display(Image(filename=OUT_PNG))

# PROVENANCE.md
if os.path.exists('PROVENANCE.md'):
    with open('PROVENANCE.md', 'r') as f:
        content = f.read()
        if 'Semaine 3' in content:
            print('\n✅ PROVENANCE.md contient section Semaine 3')
        else:
            print('\n❌ PROVENANCE.md MANQUE section Semaine 3')

print('\n✅ SEMAINE 3 TERMINÉE - Prêt pour Mois 4 (D3PM)')

# Commandes pour commit + push
print('''
POUR COMMIT + PUSH :
   !git add -A
   !git commit -m "fix(semaine3): P0 corrections - traceability, generation eval, weight tying, GPU seed"
   !git push origin main
''')